# Prédiction de la taxonomie: niveau utilisateur
Ce notebook vise à présenter l'utilisation des modèles XGBoost entraînés afin d'effectuer une classification hiérarchique des séquences de génome.

Au préalable, il faut créer l'environnement python ,voir le Readme.md partie I.  Environnement et cloner le dépot.

Pour obtenir la prédiction, vous avez besoin de :
 1. un répertoire contenant un arbre phylogénétique `phylo_tree.txt`  et des modèles xgboost entrainés pour chaque noeud interne de l'arbre, au format `.json` comme par exmple `Bacillales_order.json`. Ces fichiers sont générés lors de l'entrainement sur un dataset
 2. de paramètres d'inférence regroupés dans le yaml `predict_params.yaml`
 3. d'un fichier .fna ou d'un répertoire contenant des génomes à classifier. Un fichier .fna peut contenir plusieurs séquences qui seront chacune classifiées


In [1]:
import sys
sys.path.append('../..')
from wisp_light.prediction.predict import TaxoPredictor

In [2]:
model_dir = "/home/hcourtei/Projects/MicroTaxo/codes/exp/model_base_complete_05_15_16_37"
predict_dir = None  # permet de sauvegarder les log et les predictions
params_file = "/home/hcourtei/Projects/MicroTaxo/codes/wisp_light/prediction/predict_params.yaml"
fna_path = "/home/hcourtei/Projects/MicroTaxo/codes/data/refseq_data/GCF_000725405.1_ASM72540v1_genomic.fna"

## 0. Etapes de la prédiction

Afin d'obtenir la prédiction d'une séquence, plusieurs étapes sont nécessaires avec leur paramètre associé, à renseigner  dans le predict_param.yaml 

### a. L' échantillonnage :
  - `max_sampling=100` ou None
  - `read_size= 10_000`
  -  `ksize=4`
  - `pattern=[1,1,1,1]` 
  - `shift_ratio=None` ou un flotant entre 0 et 1

 la séquence est découpée  régulièremenent en `max_sampling=100` fenêtres de taille `read_size= 10_000` , sur chacune desquelles on dénombre  les kmers de taille `ksize=4` avec le motif `pattern=[1,1,1,1]`. Au lieu de travailer à nombre de fenêtres fixes, on peut spécifier le taux de recouvrement `shift_ratio` .  `shift_ratio=0.5` avec `read_size= 10_000` signifie que le recouvrement entre les fenetres est de `5_000 = 0.5 x 10_000`. Ces paramètres doivent être les mêmes qu'à l'entrainement des modèles

Voici les quelques lignes centrales permettant la prédiction  dans  `wisp_light.training.create_prediction`
```python
from xgboost import Booster, DMatrix
bst = Booster()
bst.load_model(model_path)
...
predictions = bst.predict(DMatrix(datas_path+"?format=libsvm"))
```
Avec les paramètres d'échantillonnage ci-dessus, pour le premier niveau de classification ('Root'), la variable `prediction` est un tenseur de taille `100 x nb_classes` ou chaque des 100 lignes est une probabilités sur les `nb_classes` classes. 

### b. Le post-traitement : 

- `read_identity_threshold=0.8`
- `normalisation_func = delta_mean`
- `threshold=0.6`

Mettons nous au 1er niveau de classification 'Root', avec 100 fenêtres et 5 classes possibles dans l'arbre phylogénétique.
la tenseur  `prediction` est de taille 100 x 5. 

A chaque fenêtre, Une solution simple serait de prendre pour classe prédite, le maximum de probabilité ( `argmax` ).
Mais l'idée mise en place dans https://github.com/dubssieg/wisp est de filtrer les prédictions trop incertaines, par un "consensus". 
Si le maximum pour une fenêtre donné, n'est pas assez distinct des autres probabilités de  classes, on ne le considère pas ensuite (classé incertaine en False).

Selon la fonction  `normalisation_func`, on  mesure cela de 3 façons différentes
- `delta_mean` (par défaut): différence entre probabilité maximum et la moyenne supérieure à `read_identity_threshold`

- `min_max` : différence entre probabilité maximum et la seconde probabilité la plus haute supérieure à `read_identity_threshold`

- `delta_sum` : probabilité maximum doit être plus grande que la somme de toutes les autres + seuil.
Si aucune des 100 fenetres n'est classée avec certitude, on baisse alors récursivement le seuil `read_identity_threshold` 
Ensuite, la classe prédite est celle qui est majoritaire en dehors de celles incertaines (False)

Sur cet exemple de sortie {'Root': {'Pseudomonadota': 40, 'Bacillota': 20, False: 30}}, 
la classe prédite est 'Pseudomonadota' : (40 vote sur 60, soit ~66%)



### c . niveau hiérachique suivant 
Ensuite, on ne explore l'arbre hiérarchique pour tous les noeuds qui sont prédit avec  au moins certain seuil de confiance `threshold=60%`. 
Dans l'expemple précédent, on ne conserverait que les noeuds provenant du taxon "Pseudomonadota" pour prédire le niveau taxnomique suivant en réitérant les étapes a. b. et c. précédentes.



 
## 1. Méthode pour une prédiction brute

In [3]:
predictor = TaxoPredictor(model_dir, params_file, predict_dir=predict_dir)
all_results = predictor.predict_one_fna(fna_path, raw_pred=True,verbose=True)


11:58 - TaxoPredictor - INFO - Params for prediction
{'ksize': 4,
 'max_sampling': 100,
 'normalisation_func': 'delta_mean',
 'pattern': [1, 1, 1, 1],
 'read_identity_threshold': 0.8,
 'read_size': 10000,
 'shift_ratio': None,
 'threshold': 0.6}
11:58 - TaxoPredictor - INFO - TaxoPredictor init for XGBoost models from directory: /home/hcourtei/Projects/MicroTaxo/codes/exp/model_base_complete_05_15_16_37
11:58 - TaxoPredictor - INFO - ============================================================
11:58 - TaxoPredictor - INFO - Prediction for GCF_000725405.1_ASM72540v1_genomic.fna contains 2 sequences.
11:58 - TaxoPredictor - INFO - Raw prediction for genome id NZ_CP008889.1 of GCF_000725405.1_ASM72540v1_genomic.fna
11:58 - TaxoPredictor - INFO - 
[{'Root': {'Actinomycetota': 84}},
 {'Actinomycetota': {'Actinomycetes': 100}},
 {'Actinomycetes': {'Actinomycetales': 1, 'Micrococcales': 95}},
 {'Micrococcales': {'Dermacoccaceae': 88, 'Micrococcaceae': 1}}]
11:58 - TaxoPredictor - INFO - Taxo 

Dans la prédiction brute, 'Root': {'Actinomycetota': 84} signifie que 84 fenêtres sont classifiés 'Actinomycetota'
et les autres ??

## 2. Résumé dans un tableau

In [4]:
predictor.results_table


,fna_file,id_seq,phylum,class,order,family
0,GCF_000725405.1_ASM72540v1_genomic.fna,NZ_CP008889.1,[{'Actinomycetota': 84}],[{'Actinomycetes': 100}],"[{'Micrococcales': 95, 'Actinomycetales': 1}]","[{'Dermacoccaceae': 88, 'Micrococcaceae': 1}]"
1,GCF_000725405.1_ASM72540v1_genomic.fna,NZ_CP008890.1,[{'Actinomycetota': 84}],[{'Actinomycetes': 100}],"[{'Micrococcales': 35, 'Actinomycetales': 6}]","[{'Dermacoccaceae': 45, 'Micrococcaceae': 26}]"


In [5]:
print(predictor.results_table.to_markdown()) # methode pour affihcer le tableau en markdown
 

|    | fna_file                               | id_seq        | phylum                   | class                    | order                                         | family                                         |
|---:|:---------------------------------------|:--------------|:-------------------------|:-------------------------|:----------------------------------------------|:-----------------------------------------------|
|  0 | GCF_000725405.1_ASM72540v1_genomic.fna | NZ_CP008889.1 | [{'Actinomycetota': 84}] | [{'Actinomycetes': 100}] | [{'Micrococcales': 95, 'Actinomycetales': 1}] | [{'Dermacoccaceae': 88, 'Micrococcaceae': 1}]  |
|  1 | GCF_000725405.1_ASM72540v1_genomic.fna | NZ_CP008890.1 | [{'Actinomycetota': 84}] | [{'Actinomycetes': 100}] | [{'Micrococcales': 35, 'Actinomycetales': 6}] | [{'Dermacoccaceae': 45, 'Micrococcaceae': 26}] |


## 3. Extraction de la Prediction majoritaire

In [6]:
all_results = predictor.predict_one_fna(fna_path, raw_pred=False, verbose=False)
predictor.results_table 


,fna_file,id_seq,phylum,class,order,family
0,GCF_000725405.1_ASM72540v1_genomic.fna,NZ_CP008889.1,[{'Actinomycetota': 84}],[{'Actinomycetes': 100}],"[{'Micrococcales': 95, 'Actinomycetales': 1}]","[{'Dermacoccaceae': 88, 'Micrococcaceae': 1}]"
1,GCF_000725405.1_ASM72540v1_genomic.fna,NZ_CP008890.1,[{'Actinomycetota': 84}],[{'Actinomycetes': 100}],"[{'Micrococcales': 35, 'Actinomycetales': 6}]","[{'Dermacoccaceae': 45, 'Micrococcaceae': 26}]"
2,GCF_000725405.1_ASM72540v1_genomic.fna,NZ_CP008889.1,Actinomycetota,Actinomycetes,Micrococcales,Dermacoccaceae
3,GCF_000725405.1_ASM72540v1_genomic.fna,NZ_CP008890.1,Actinomycetota,Actinomycetes,Micrococcales,Dermacoccaceae


## 4. Prediction sur un répertoire de fichiers .fna
Une mention "Seq too short"  est faite sur la classification lorsque les séquences sont trop courtes , ici < `read_size= 10_000`

In [7]:
fna_dir = "/home/hcourtei/Projects/MicroTaxo/codes/predict/to_predict"
predictor = TaxoPredictor(model_dir, params_file, predict_dir=None)
predictor.predict_one_dir(fna_dir, raw_pred=False, verbose=False)
predictor.results_table

11:58 - TaxoPredictor - INFO - Params for prediction
{'ksize': 4,
 'max_sampling': 100,
 'normalisation_func': 'delta_mean',
 'pattern': [1, 1, 1, 1],
 'read_identity_threshold': 0.8,
 'read_size': 10000,
 'shift_ratio': None,
 'threshold': 0.6}
11:58 - TaxoPredictor - INFO - TaxoPredictor init for XGBoost models from directory: /home/hcourtei/Projects/MicroTaxo/codes/exp/model_base_complete_05_15_16_37
11:58 - TaxoPredictor - INFO - ============================================================
11:58 - TaxoPredictor - INFO - Prediction for whole dir /home/hcourtei/Projects/MicroTaxo/codes/predict/to_predict,  containing 5 fna files.


100%|██████████| 5/5 [00:19<00:00,  3.82s/it]


,fna_file,id_seq,phylum,class,order,family
0,GCF_000737865.1_ASM73786v1_genomic.fna,NZ_CP007287.1,Actinomycetota,Actinomycetes,Bifidobacteriales,Bifidobacteriaceae
1,GCF_000741575.1_Bifcun_genomic.fna,NZ_JGYV01000001.1,Actinomycetota,Actinomycetes,Bifidobacteriales,Bifidobacteriaceae
2,GCF_000741575.1_Bifcun_genomic.fna,NZ_JGYV01000002.1,Actinomycetota,Actinomycetes,Bifidobacteriales,Bifidobacteriaceae
3,GCF_000741575.1_Bifcun_genomic.fna,NZ_JGYV01000003.1,Seq too short,Seq too short,Seq too short,Seq too short
4,GCF_000741575.1_Bifcun_genomic.fna,NZ_JGYV01000004.1,Actinomycetota,Actinomycetes,Bifidobacteriales,Bifidobacteriaceae
...,...,...,...,...,...,...
88,GCF_000730685.1_ASM73068v1_genomic.fna,NZ_BBIX01000005.1,Pseudomonadota,Gammaproteobacteria,Pasteurellales,Pasteurellaceae
89,GCF_000730685.1_ASM73068v1_genomic.fna,NZ_BBIX01000004.1,Pseudomonadota,Gammaproteobacteria,Pasteurellales,Pasteurellaceae
90,GCF_000730685.1_ASM73068v1_genomic.fna,NZ_BBIX01000003.1,Pseudomonadota,Gammaproteobacteria,Pasteurellales,Pasteurellaceae
91,GCF_000730685.1_ASM73068v1_genomic.fna,NZ_BBIX01000002.1,Pseudomonadota,Gammaproteobacteria,Pasteurellales,Pasteurellaceae
